# <font style = "color:rgb(50,120,229)">Skin Smoothing</font>

In previous modules we have seen how we can use grabcut for detecting skin region given the image of a face. The detected skin can then be smoothened and applied back to the original image, thereby resulting in a skin smoothened image. 

In this project, you will be implementing **Skin Smoothing** but this time the image will contain regions other than face as well and will have to be completely automated.

You can use the following steps to approach this problem:

1. Detect the faces in the image using Deep Learning or HAAR Cascades
2. Iterate over the detected faces and apply smoothing filter. You can experiment with the filter type and size to see which one (or combination) gives the best result.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from dataPath import DATA_PATH
%matplotlib inline

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (20.0, 10.0)
matplotlib.rcParams['image.cmap'] = 'gray'

In [ ]:
modelFile = DATA_PATH + "models/opencv_face_detector_uint8.pb"
configFile = DATA_PATH + "models/opencv_face_detector.pbtxt"
net = cv2.dnn.readNetFromTensorflow(modelFile, configFile)

In [ ]:
def detectFaceOpenCVDnn(net, image, conf_threshold=0.9):
    imageOpencvDnn = image.copy()
    imageHeight = imageOpencvDnn.shape[0]
    imageWidth = imageOpencvDnn.shape[1]
    blob = cv2.dnn.blobFromImage(imageOpencvDnn, 1.0, (300, 300), [104, 117, 123], False, False)

    net.setInput(blob)
    detections = net.forward()
    bboxes = []
    for i in range(detections.shape[2])[:1]:
        confidence = detections[0, 0, i, 2]
        if confidence > conf_threshold:
            x1 = int(detections[0, 0, i, 3] * imageWidth)
            y1 = int(detections[0, 0, i, 4] * imageHeight)
            x2 = int(detections[0, 0, i, 5] * imageWidth)
            y2 = int(detections[0, 0, i, 6] * imageHeight)
            bboxes.append([x1, y1, x2, y2])
            cv2.rectangle(imageOpencvDnn, (x1, y1), (x2, y2), (0, 255, 0), int(round(imageHeight/150)), 8)
    return imageOpencvDnn, bboxes

In [ ]:
filenames = [DATA_PATH + 'images/hillary_clinton.jpg', DATA_PATH + 'images/obama.jpg', DATA_PATH + 'images/trump.jpg']
images = [cv2.imread(filename) for filename in filenames]

In [ ]:
for image in images:
    plt.subplot(121)
    plt.title("Original Image")
    plt.imshow(image[:,:,::-1])
    output, bboxes = detectFaceOpenCVDnn(net, image)
    if len(bboxes) > 0:
        x1, y1, x2, y2 = bboxes[0]
        face = image[y1:y2, x1:x2].copy()
        height, width = face.shape[:2]
        
        # Forehead
        forehead_rel_loc = [0.1, 0.3, 0.3, 0.7] # y1, y2, x1, x2
        forehead_abs_loc = [int(forehead_rel_loc[0] * height), int(forehead_rel_loc[1] * height), int(forehead_rel_loc[2] * width), int(forehead_rel_loc[3] * width)]
        forehead = face[forehead_abs_loc[0]:forehead_abs_loc[1], forehead_abs_loc[2]:forehead_abs_loc[3]]
        # Left cheek
        lcheek_rel_loc = [0.5, 0.6, 0.1, 0.3] # y1, y2, x1, x2
        lcheek_abs_loc = [int(lcheek_rel_loc[0] * height), int(lcheek_rel_loc[1] * height), int(lcheek_rel_loc[2] * width), int(lcheek_rel_loc[3] * width)]
        lcheek = face[lcheek_abs_loc[0]:lcheek_abs_loc[1], lcheek_abs_loc[2]:lcheek_abs_loc[3]]
        # Right cheek
        rcheek_rel_loc = [0.5, 0.6, 0.7, 0.9] # y1, y2, x1, x2
        rcheek_abs_loc = [int(rcheek_rel_loc[0] * height), int(rcheek_rel_loc[1] * height), int(rcheek_rel_loc[2] * width), int(rcheek_rel_loc[3] * width)]
        rcheek = face[rcheek_abs_loc[0]:rcheek_abs_loc[1], rcheek_abs_loc[2]:rcheek_abs_loc[3]]
        
        # Skin image
        skin_image = np.hstack([lcheek, rcheek])
        skin_image_width = np.minimum(skin_image.shape[1], forehead.shape[1])
        skin_image = skin_image[:, :skin_image_width, :]
        forehead = forehead[:, :skin_image_width, :]
        skin_image = np.vstack([forehead, skin_image])
        
        # Skin color model
        skin_image_hsv = cv2.cvtColor(skin_image, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(skin_image_hsv)
        q = [0.01, 0.99]
        h_lower, h_upper = np.quantile(h, q)
        s_lower, s_upper = np.quantile(s, q)
        v_lower, v_upper = np.quantile(v, q)
        lowerb = np.array([h_lower, s_lower, v_lower])
        upperb = np.array([h_upper, s_upper, v_upper])
    
        # Skin mask
        face_hsv = cv2.cvtColor(face, cv2.COLOR_BGR2HSV)
        skin_mask = cv2.inRange(face_hsv, lowerb, upperb)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        skin_mask = cv2.dilate(skin_mask, kernel, iterations=1)
        skin_mask = cv2.erode(skin_mask, kernel, iterations=1)
        skin_mask = np.uint8(skin_mask / 255)
        skin_mask = cv2.merge((skin_mask, skin_mask, skin_mask))
    
        # Face blur
        face_blur = cv2.bilateralFilter(face, 9, 50, 50)
        
        # Merge smoothed skin
        face = cv2.multiply(face, (1 - skin_mask))
        face_blur = cv2.multiply(face_blur, skin_mask)
        face = cv2.add(face, face_blur)
        
        # Replace face with smoothed skin in original image
        result = image.copy()
        result[y1:y2, x1:x2] = face        
        
        plt.subplot(122)
        plt.title("Revised Image")
        plt.imshow(result[:,:,::-1])
        
    plt.show()